In [ ]:
from pathlib import Path
import re
import pandas as pd
import marimo as mo

In [ ]:
# Put messy_ecommerce_orders.csv and messy_customers.csv in the same
# folder as this notebook, or keep them under dataset-source/data/raw/.
candidates = [
    Path("."),
    Path("dataset-source/data/raw"),
    Path("data/raw"),
]
data_dir = next(
    (
        folder
        for folder in candidates
        if (folder / "messy_ecommerce_orders.csv").exists()
        and (folder / "messy_customers.csv").exists()
    ),
    None,
)

if data_dir is None:
    raise FileNotFoundError(
        "Could not find both CSV files. Place messy_ecommerce_orders.csv "
        "and messy_customers.csv beside this notebook, or in "
        "dataset-source/data/raw/."
    )

orders_raw = pd.read_csv(data_dir / "messy_ecommerce_orders.csv")
customers_raw = pd.read_csv(data_dir / "messy_customers.csv")

In [ ]:
mo.md(f"""
# E-commerce data cleaning and preprocessing

**Data source folder:** `{data_dir}`

This notebook audits the raw data, removes exact duplicate rows,
standardizes text and dates, validates prices and quantities, and
calculates the highest-priced product and revenue by city.

- Orders loaded: **{len(orders_raw):,} rows**
- Customers loaded: **{len(customers_raw):,} rows**
""")

# E-commerce data cleaning and preprocessing

**Data source folder:** `dataset-source/data/raw`

This notebook audits the raw data, removes exact duplicate rows,
standardizes text and dates, validates prices and quantities, and
calculates the highest-priced product and revenue by city.

- Orders loaded: **213 rows**
- Customers loaded: **125 rows**

In [ ]:
raw_overview = mo.md(f"""
    ## 1. Inspect the raw data

    **Orders shape:** `{orders_raw.shape}`

    **Customer shape:** `{customers_raw.shape}`
    """)
raw_overview

## 1. Inspect the raw data

**Orders shape:** `(213, 11)`

**Customer shape:** `(125, 5)`

In [ ]:
orders_raw.head(10), customers_raw.head(10)

In [ ]:
# Orders Table Details

print(orders_raw.columns)
print(customers_raw.columns)

Index(['Order ID', 'customer_id', 'Purchase Date', 'delivery_date',
       'product_category', 'price', 'Quantity', 'payment_type',
       'customer_city', 'review_score', 'review_comment'],
      dtype='str')
Index(['customer_id', 'customer_name', 'email', 'signup_date',
       'customer_city'],
      dtype='str')


In [ ]:
orders_raw.info(), customers_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order ID          210 non-null    str    
 1   customer_id       210 non-null    str    
 2   Purchase Date     210 non-null    str    
 3   delivery_date     192 non-null    str    
 4   product_category  210 non-null    str    
 5   price             205 non-null    str    
 6   Quantity          209 non-null    str    
 7   payment_type      210 non-null    str    
 8   customer_city     210 non-null    str    
 9   review_score      195 non-null    float64
 10  review_comment    54 non-null     str    
dtypes: float64(1), str(10)
memory usage: 33.7 KB
<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    125 non-null    str  
 1   customer_name  125 no

In [ ]:
def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )
    return df

orders_raw[:] = standardize_columns(orders_raw)
customers_raw[:] = standardize_columns(customers_raw)

print(orders_raw.columns)
print(customers_raw.columns)

Index(['order_id', 'customer_id', 'purchase_date', 'delivery_date',
       'product_category', 'price', 'quantity', 'payment_type',
       'customer_city', 'review_score', 'review_comment'],
      dtype='str')
Index(['customer_id', 'customer_name', 'email', 'signup_date',
       'customer_city'],
      dtype='str')


In [ ]:
# Missing values statistics
def missing_report(df, table_name):
    result = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": [df[col].isna().sum() for col in df.columns],
        }
    )
    result["missing_percent"] = (
        (result["missing_count"] / len(df) * 100).round(2) if len(df) else 0
    )
    result.insert(0, "table", table_name)
    return result.sort_values("missing_count", ascending=False)

missing_values = pd.concat(
    [
        missing_report(orders_raw, "orders"),
        missing_report(customers_raw, "customers"),
    ],
    ignore_index=True,
)
missing_values

,table,column,missing_count,missing_percent
0,orders,review_comment,159,74.65
1,orders,delivery_date,21,9.86
2,orders,review_score,18,8.45
3,orders,price,8,3.76
4,orders,quantity,4,1.88
...,...,...,...,...
11,customers,email,3,2.40
12,customers,customer_id,0,0.00
13,customers,customer_name,0,0.00
14,customers,signup_date,0,0.00


In [ ]:
# Dulicates statistics
duplicate_row_count = int(orders_raw.duplicated().sum())
orders_no_duplicates = orders_raw.drop_duplicates().copy()

duplicate_summary = pd.DataFrame(
    {
        "metric": ["Rows before", "Exact duplicate rows", "Rows after"],
        "count": [
            len(orders_raw),
            duplicate_row_count,
            len(orders_no_duplicates),
        ],
    }
)
duplicate_summary

,metric,count
0,Rows before,213
1,Exact duplicate rows,15
2,Rows after,198


In [ ]:
# Perform string normalization to orders
def clean_text_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
        df[col] = df[col].replace("", pd.NA)
    return df

orders_text_clean = clean_text_columns(orders_no_duplicates)

In [ ]:
# Perform string normalization to customers
def clean_customer_text(df):
    df = df.copy()
    df.columns = df.columns.str.strip()
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
        df[col] = df[col].replace("", pd.NA)
    return df

customers_text_clean = clean_customer_text(customers_raw)

In [ ]:
# Format convertion for string type columns
for col in ["product_category", "payment_type", "customer_city"]:
    if col in orders_text_clean.columns:
        print(f"\n{col}:")
        print(
            orders_text_clean[col]
            .value_counts(dropna=False)
            .to_string()
        )

print()

if "customer_city" in customers_text_clean.columns:
    print(
        customers_text_clean["customer_city"]
        .value_counts(dropna=False)
        .to_string()
    )


product_category:
product_category
Toys           24
electronics    24
clothing       24
Electronics    21
furniture      21
Sports         19
Books          19
Clothing       17
Furniture      14
ELECTRONICS    14
<NA>            1

payment_type:
payment_type
Debit Card     35
Credit Card    27
paypal         27
cash           25
PayPal         23
debit card     23
credit_card    20
credit card    17
<NA>            1

customer_city:
customer_city
belo horizonte     28
Curitiba           24
sao paulo          20
curitiba           20
SAO PAULO          16
Salvador           15
Rio de Janeiro     14
Sao Paulo          10
Belo Horizonte      9
rio de janeiro      9
BrasÃ­lia           4
S√£o Paulo          3
​sao paulo          3
​curitiba           3
Brasília            3
​ Salvador          3
São Paulo           2
​belo horizonte     2
​Sao Paulo          2
​Belo Horizonte     2
​rio de janeiro     2
​Rio de Janeiro     2
<NA>                1
​SAO PAULO          1

customer_city
SAO

In [ ]:
category_map = {
    "toys": "Toys",
    "electronics": "Electronics",
    "clothing": "Clothing",
    "furniture": "Furniture",
    "sports": "Sports",
    "books": "Books",
}
payment_map = {
    "debit card": "Debit Card",
    "credit card": "Credit Card",
    "credit_card": "Credit Card",
    "paypal": "PayPal",
    "cash": "Cash",
}

orders_standardized = orders_text_clean.copy()
orders_standardized["product_category"] = (
    orders_standardized["product_category"].astype("string").str.casefold().map(category_map)
)
orders_standardized["payment_type"] = (
    orders_standardized["payment_type"].astype("string").str.casefold().map(payment_map)
)

def standardize_city(series):
    city = (
        series.astype("string")
        .str.replace("\u200b", "", regex=False)
        .str.replace("\ufeff", "", regex=False)
        .str.replace("\xa0", " ", regex=False)
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )
    city_map = {
        "sao paulo": "São Paulo",
        "são paulo": "São Paulo",
        "s√£o paulo": "São Paulo",
        "belo horizonte": "Belo Horizonte",
        "curitiba": "Curitiba",
        "rio de janeiro": "Rio de Janeiro",
        "salvador": "Salvador",
        "brasília": "Brasília",
        "brasilia": "Brasília",
        "brasã­lia": "Brasília",
    }
    return city.map(city_map)

orders_standardized["customer_city"] = standardize_city(orders_standardized["customer_city"])
customers_preprocessed = customers_text_clean.copy()
customers_preprocessed["customer_city"] = standardize_city(
    customers_preprocessed["customer_city"]
)

category_counts = orders_standardized["product_category"].value_counts(dropna=False)
payment_counts = orders_standardized["payment_type"].value_counts(dropna=False)
city_counts = orders_standardized["customer_city"].value_counts(dropna=False)

category_counts, payment_counts, city_counts

In [ ]:
# Parse mixed date formats explicitly where possible.
# Slash dates are treated as MM/DD/YYYY to match examples such as 01/25/2024.
def parse_mixed_date(value):
    if pd.isna(value):
        return pd.NaT
    value = str(value).strip()
    if value.casefold() in {"", "?", "--", "unknown", "nan", "nat"}:
        return pd.NaT
    if re.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}", value):
        return pd.to_datetime(
            value, format="%Y-%m-%d", errors="coerce"
        ).normalize().strftime("%Y-%m-%d")
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", value):
        return pd.to_datetime(
            value, format="%m/%d/%Y",
            errors="coerce"
        ).normalize().strftime("%Y-%m-%d")
    if re.fullmatch(r"\d{1,2}-\d{1,2}-\d{4}", value):
        return pd.to_datetime(
            value, format="%d-%m-%Y",
            errors="coerce"
        ).normalize().strftime("%Y-%m-%d")
    return pd.to_datetime(
        value, errors="coerce"
    ).normalize().strftime("%Y-%m-%d")

orders_dated = orders_standardized.copy()
for date_col in ["purchase_date", "delivery_date"]:
    if date_col in orders_dated.columns:
        orders_dated[date_col] = orders_dated[date_col].map(parse_mixed_date)

customers_dated = customers_preprocessed.copy()
if "signup_date" in customers_dated.columns:
    customers_dated["signup_date"] = customers_dated["signup_date"].map(parse_mixed_date)

date_quality = pd.DataFrame(
    {
        "column": ["purchase_date", "delivery_date"],
        "missing_after_parsing": [
            orders_dated["purchase_date"].isna().sum(),
            orders_dated["delivery_date"].isna().sum(),
        ],
    }
)
date_quality

,column,missing_after_parsing
0,purchase_date,1
1,delivery_date,19


In [ ]:
orders_dated[['purchase_date', 'delivery_date']], customers_dated[['signup_date']]

In [ ]:
# Clean prices: remove currency symbols/commas and turn placeholders into NA.
orders_cleaned = orders_dated.copy()
price_text = (
    orders_cleaned["price"]
    .astype("string")
    .str.strip()
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .replace(["", "--", "unknown", "Unknown", "?", "nan"], pd.NA)
)
orders_cleaned["price"] = pd.to_numeric(price_text, errors="coerce")

# Clean quantities: convert numeric strings; placeholders become NA.
quantity_text = (
    orders_cleaned["quantity"]
    .astype("string")
    .str.strip()
    .replace(["", "--", "unknown", "Unknown", "?", "nan"], pd.NA)
)
orders_cleaned["quantity"] = pd.to_numeric(quantity_text, errors="coerce")

numeric_quality = pd.DataFrame(
    {
        "check": [
            "Missing/invalid prices",
            "Non-positive prices",
            "Missing/invalid quantities",
            "Non-positive quantities",
        ],
        "count": [
            int(orders_cleaned["price"].isna().sum()),
            int((orders_cleaned["price"].notna() & orders_cleaned["price"].le(0)).sum()),
            int(orders_cleaned["quantity"].isna().sum()),
            int((orders_cleaned["quantity"].notna() & orders_cleaned["quantity"].le(0)).sum()),
        ],
    }
)
numeric_quality

,check,count
0,Missing/invalid prices,11
1,Non-positive prices,6
2,Missing/invalid quantities,5
3,Non-positive quantities,9


In [ ]:
invalid_numeric_rows = orders_cleaned.loc[
    (orders_cleaned["price"].le(0) & orders_cleaned["price"].notna())
    | (orders_cleaned["quantity"].le(0) & orders_cleaned["quantity"].notna())
]

invalid_numeric_rows[["order_id", "price", "quantity"]]

,order_id,price,quantity
13,ORD1109,444.24,-1
23,ORD1035,401.71,-1
62,ORD1031,307.03,-4
77,ORD1193,12.8,-3
93,ORD1176,-222.86,1
...,...,...,...
187,ORD1061,-124.47,1
192,ORD1027,-119.56,1
203,ORD1108,-394.06,-1
205,ORD1148,-40.72,1


In [ ]:
# Highest-priced product among rows with a known positive price.
valid_prices = orders_cleaned.loc[
    orders_cleaned["price"].notna() & orders_cleaned["price"].gt(0)
]

if valid_prices.empty:
    highest_price_record = pd.DataFrame(
        columns=["Order ID", "product_category", "price", "customer_city"]
    )
else:
    max_price = valid_prices["price"].max()
    highest_price_record = valid_prices.loc[
        valid_prices["price"].eq(max_price),
        ["order_id", "product_category", "price", "customer_city"],
    ]

highest_price_record

,order_id,product_category,price,customer_city
128,ORD1194,Sports,9692.07,São Paulo


In [ ]:
# Revenue = price * quantity. For this conservative sales summary, include
# only rows with a known positive price and positive quantity and known city.
# Negative/zero quantities are excluded pending confirmation whether they
# represent returns, cancellations, or dirty values.
valid_sales = orders_cleaned.loc[
    orders_cleaned["price"].notna()
    & orders_cleaned["price"].gt(0)
    & orders_cleaned["quantity"].notna()
    & orders_cleaned["quantity"].gt(0)
    & orders_cleaned["customer_city"].notna()
].copy()

valid_sales["revenue"] = valid_sales["price"] * valid_sales["quantity"]

city_revenue = (
    valid_sales.groupby("customer_city", as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
    .reset_index(drop=True)
)
city_revenue["revenue"] = city_revenue["revenue"].round(2)

revenue_summary = pd.DataFrame(
    {
        "metric": [
            "Rows after exact-deduplication",
            "Rows included in revenue calculation",
            "Rows excluded from revenue calculation",
            "Total revenue from included rows",
        ],
        "value": [
            len(orders_cleaned),
            len(valid_sales),
            len(orders_cleaned) - len(valid_sales),
            round(valid_sales["revenue"].sum(), 2),
        ],
    }
)

revenue_summary, city_revenue

In [ ]:
# Optional customer/order consistency check. This does not delete records.
order_customer_ids = set(orders_cleaned["customer_id"].dropna().astype(str))
customer_ids = set(customers_dated["customer_id"].dropna().astype(str))
unmatched_order_customers = sorted(order_customer_ids - customer_ids)

customer_join_check = pd.DataFrame(
    {
        "metric": [
            "Distinct customer IDs in orders",
            "Distinct customer IDs in customer table",
            "Order customer IDs absent from customer table",
        ],
        "count": [
            len(order_customer_ids),
            len(customer_ids),
            len(unmatched_order_customers),
        ],
    }
)
customer_join_check, unmatched_order_customers

## 7. Findings and recommended actions

- **Missing values:** review the missing-value table above. Do not fill
  fields such as delivery dates or review comments with invented data.
- **Duplicates:** only exact duplicate rows were removed. Repeated order
  IDs alone were retained because an order may contain multiple items.
- **Prices and quantities:** inspect the invalid-record tables. Confirm
  whether non-positive quantities represent returns before deciding
  whether to exclude or model them separately.
- **Revenue:** the city summary includes only rows with positive prices,
  positive quantities, and a known city. It is a conservative estimate
  of revenue from valid sale rows, not a complete accounting ledger.
- **Possible root causes:** inconsistent data entry, mixed date formats,
  placeholder strings (`--`, `?`, `unknown`), inconsistent capitalization,
  encoding problems, missing form fields, and duplicate imports.
- **Prevention:** enforce schema/type validation at ingestion, standardize
  city/category/payment values, require appropriate fields, and monitor
  data quality before reporting.

In [ ]:
# Export cleaned tables and analysis outputs beside the notebook.
output_dir = Path("cleaned_output")
output_dir.mkdir(exist_ok=True)

orders_cleaned.to_csv(output_dir / "orders_cleaned.csv", index=False)
customers_dated.to_csv(output_dir / "customers_cleaned.csv", index=False)
city_revenue.to_csv(output_dir / "revenue_by_city.csv", index=False)

f"Saved cleaned CSVs and revenue report to: {output_dir.resolve()}"

'Saved cleaned CSVs and revenue report to: /Users/sasidharana/development/DataAnalysis/cleaned_output'

In [ ]:
orders_cleaned

,order_id,customer_id,purchase_date,delivery_date,product_category,price,quantity,payment_type,customer_city,review_score,review_comment
0,ORD1175,CUST197,2024-01-25,2024-02-07,Electronics,295.63,1,PayPal,São Paulo,4.0,<NA>
1,ORD1103,CUST234,2024-01-28,2024-02-02,Furniture,<NA>,4,Cash,Rio de Janeiro,NaN,<NA>
2,ORD1134,CUST220,2024-02-08,2024-02-10,Furniture,269.7,4,Debit Card,São Paulo,3.0,<NA>
3,ORD1169,CUST132,2024-09-26,<NA>,Toys,283.9,1,Credit Card,Belo Horizonte,5.0,<NA>
4,ORD1104,CUST167,2024-03-18,2024-04-05,Electronics,167.89,4,Debit Card,Rio de Janeiro,4.0,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
206,ORD1022,CUST180,2024-03-30,2024-04-01,Electronics,137.74,4,Credit Card,São Paulo,5.0,<NA>
207,ORD1001,CUST202,2024-09-27,2024-10-04,Electronics,228.46,-3,Credit Card,Rio de Janeiro,3.0,Not as expected
210,ORD1132,CUST144,2024-01-01,2024-01-09,Sports,454.73,3,PayPal,São Paulo,5.0,<NA>
211,ORD1153,CUST215,2024-01-24,2024-01-30,Electronics,383.53,<NA>,Debit Card,Rio de Janeiro,3.0,Fast delivery
